In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # Use GPU parse_dates to avoid post-read conversions
    df = pd.read_csv(
        data_path,
        sep="|",
        storage_options=storage_options,
        parse_dates=["L_SHIPDATE", "L_RECEIPTDATE", "L_COMMITDATE"]
    )
    print(df.columns)
    return df

lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df

supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

lineitem_filtered = lineitem[
    (lineitem["L_SHIPDATE"] >= pd.Timestamp("1996-01-01"))
    & (
        lineitem["L_SHIPDATE"]
        < (pd.Timestamp("1996-01-01") + pd.DateOffset(months=3))
    )
]
lineitem_filtered["REVENUE_PARTS"] = lineitem_filtered["L_EXTENDEDPRICE"] * (
    1.0 - lineitem_filtered["L_DISCOUNT"]
)
lineitem_filtered = lineitem_filtered.loc[:, ["L_SUPPKEY", "REVENUE_PARTS"]]
revenue_table = (
    lineitem_filtered.groupby("L_SUPPKEY", as_index=False, sort=False)
    .agg(TOTAL_REVENUE=pd.NamedAgg(column="REVENUE_PARTS", aggfunc="sum"))
    .rename(columns={"L_SUPPKEY": "SUPPLIER_NO"})
)
max_revenue = revenue_table["TOTAL_REVENUE"].max()
revenue_table = revenue_table[revenue_table["TOTAL_REVENUE"] == max_revenue]
supplier_filtered = supplier.loc[:, ["S_SUPPKEY", "S_NAME", "S_ADDRESS", "S_PHONE"]]
total = supplier_filtered.merge(
    revenue_table, left_on="S_SUPPKEY", right_on="SUPPLIER_NO", how="inner"
)
total = total.loc[
    :, ["S_SUPPKEY", "S_NAME", "S_ADDRESS", "S_PHONE", "TOTAL_REVENUE"]
]